# 🏠 House Price Prediction — Exploratory Data Analysis

This notebook provides an interactive walkthrough of the dataset.
For automated analysis, run `python main.py` which generates all plots to `outputs/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Dataset

In [ ]:
from src.generate_dataset import generate_housing_dataset

df = generate_housing_dataset(output_path='../data/raw/house_prices.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Dataset Overview

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Missing Values

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing_pct = (missing / len(df) * 100).round(1)

pd.DataFrame({'Count': missing, 'Percent': missing_pct})

## 4. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['SalePrice'], kde=True, bins=50, ax=axes[0], color='#4C72B0')
axes[0].set_title('SalePrice Distribution')
sns.histplot(np.log1p(df['SalePrice']), kde=True, bins=50, ax=axes[1], color='#DD8452')
axes[1].set_title('Log(SalePrice) Distribution')
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
numeric = df.select_dtypes(include=[np.number])
corr = numeric.corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Top Correlated Features

In [ ]:
top_corr = corr['SalePrice'].abs().sort_values(ascending=False).head(11)
print(top_corr)

## 7. Feature vs. SalePrice Scatter Plots

In [ ]:
top_features = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'FullBath', 'YearBuilt']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, feat in zip(axes.flatten(), top_features):
    ax.scatter(df[feat], df['SalePrice'], alpha=0.3, s=10)
    ax.set_xlabel(feat)
    ax.set_ylabel('SalePrice')
    ax.set_title(f'{feat} vs SalePrice')
plt.tight_layout()
plt.show()

## 8. Feature Engineering Preview

In [ ]:
from src.feature_engineering import add_engineered_features

df_eng = add_engineered_features(df.drop(columns=['SalePrice']))
new_cols = ['HouseAge', 'RemodAge', 'TotalSF', 'TotalBath', 'TotalPorchSF', 'HasPool', 'HasGarage', 'QualLivArea', 'GarageInteraction']
df_eng[new_cols].describe()